# 05 · The context window
Level L1 · Part 1 · core · ~20 min · builds on 03

## Goal
By the end of this lesson you can work out whether a chat fits a model's context window
(input + reserved output, including thinking), say what Anthropic, OpenAI and Gemini do when it does not,
and explain why a fuller window gives worse answers even before the limit ("context rot").

## The idea
A model reads one block of tokens and writes more tokens after it. The **context window** is the most
tokens that block can hold: the system prompt, the whole chat history, the new question **and** the answer
being written (with any hidden "thinking"). It is measured in tokens (lesson 03), not words or messages.
The model has no other memory: anything that is not in the window does not exist for it.
When a chat grows past the window, something has to give. Real APIs mostly **refuse with an error**;
some apps quietly drop the oldest turns. And long before the hard limit, answers get worse.

Analogy: a whiteboard of fixed size. You must leave space for the answer, and a crowded board is hard to read.

## Picture
```mermaid
flowchart LR
    subgraph W["Context window: 200 tokens"]
        S[System prompt] --> H[Chat history] --> Q[New question] --> O["Reserved output<br/>thinking + answer"]
    end
    H -. "too much?" .-> E["API: 400 error<br/>app: drop oldest turns"]
```

## Step by step
### 1. A toy model with a 200-token window
We give Scout a tiny window so we can watch it fill. To count tokens we reuse lesson 03's word tokenizer
(words and punctuation) plus 1 token per message for its role marker, because real chat formats
add a few tokens per message too. A real tokenizer would give different numbers; the mechanics are the same.

In [1]:
import re

WINDOW = 200        # most tokens the toy model can hold: input + output
MAX_OUTPUT = 40     # tokens we reserve for the reply (max output tokens)

def word_tokenize(text: str) -> list[str]:
    """Lesson 02/03's toy tokenizer: words and single punctuation marks."""
    return re.findall(r"[\w']+|[.,!?]", text)

def count_tokens(messages: list[dict]) -> int:
    return sum(1 + len(word_tokenize(m["text"])) for m in messages)

SYSTEM = {"role": "system", "text": "You are Scout. Answer only from the user's notes and this chat."}
print("system prompt:", count_tokens([SYSTEM]), "tokens")

system prompt: 15 tokens


### 2. Scout's growing chat
Here is a scripted chat about Scout's six notes. In turn 1 the user says what their project is.
Each turn adds a question and an answer, and the model is sent **everything so far** every time.

In [2]:
TURNS = [
    ("Hi Scout. My project is a school talk about the moons of Mars, for my astronomy class.",
     "Great topic. From note 1, Mars has two small moons, Phobos and Deimos."),
    ("How long is a day on Mars?", "From note 3, a day on Mars lasts about 24 hours and 39 minutes."),
    ("Which rover landed on Mars in 2021?", "From note 5, the rover Perseverance landed on Mars in February 2021."),
    ("What is the largest planet?", "From note 4, Jupiter is the largest planet in the solar system."),
    ("Tell me something odd about Venus.", "From note 2, Venus spins backwards compared with most planets."),
    ("What are Saturn's rings made of?", "From note 6, Saturn's rings are mostly made of ice."),
]
history = [SYSTEM]
for i, (question, answer) in enumerate(TURNS, start=1):
    history += [{"role": "user", "text": question, "turn": i},
                {"role": "assistant", "text": answer, "turn": i}]
    print(f"after turn {i}: {count_tokens(history):3} tokens in the chat")

after turn 1:  54 tokens in the chat
after turn 2:  80 tokens in the chat
after turn 3: 104 tokens in the chat
after turn 4: 126 tokens in the chat
after turn 5: 147 tokens in the chat
after turn 6: 168 tokens in the chat


The chat only ever grows. Now the user asks turn 7, "Remind me: what is my project about?".
The input is the history plus that question, and we must still leave `MAX_OUTPUT` tokens for the reply.

In [3]:
question7 = {"role": "user", "text": "Remind me: what is my project about?", "turn": 7}
needed = count_tokens(history + [question7]) + MAX_OUTPUT
print(f"input {count_tokens(history + [question7])} + output {MAX_OUTPUT} = {needed} tokens; window = {WINDOW}")
print("fits" if needed <= WINDOW else f"over by {needed - WINDOW} tokens")

input 177 + output 40 = 217 tokens; window = 200
over by 17 tokens


It does not fit. There are two ways to react: drop old turns to make room, or refuse. We try both.

### 3. Option A: drop the oldest turns (a sliding window)
Keep the system prompt, then add messages from the newest backwards while they fit in `WINDOW - MAX_OUTPUT`.

In [4]:
def sliding_window(messages: list[dict], window: int, max_output: int) -> list[dict]:
    system, rest = messages[0], messages[1:]
    kept, budget = [], window - max_output - count_tokens([system])
    for m in reversed(rest):                     # newest first
        cost = count_tokens([m])
        if cost > budget:
            break
        kept.insert(0, m)
        budget -= cost
    return [system] + kept

visible = sliding_window(history + [question7], WINDOW, MAX_OUTPUT)
dropped = [m for m in history + [question7] if m not in visible]
print("tokens sent:", count_tokens(visible), "of", count_tokens(history + [question7]))
for m in dropped:
    print(f"dropped (turn {m['turn']}, {m['role']}): {m['text']}")

tokens sent: 156 of 177
dropped (turn 1, user): Hi Scout. My project is a school talk about the moons of Mars, for my astronomy class.


The oldest message, the user's half of turn 1, fell out of the window. Only one message of the turn was
dropped, so Scout's reply to it is still there, now without its question. Real apps often drop whole turns
to avoid this. Our toy Scout answers from what it can see, like a real model:
it looks for a message where the user said "My project is".

In [5]:
def toy_scout(messages: list[dict]) -> str:
    for m in messages:
        if m["role"] == "user" and "My project is" in m["text"]:
            return "Your project is" + m["text"].split("My project is")[1]
    return "I don't know what your project is. Could you tell me?"

print("full history :", toy_scout(history + [question7]))
print("sliding view :", toy_scout(visible))

full history : Your project is a school talk about the moons of Mars, for my astronomy class.
sliding view : I don't know what your project is. Could you tell me?


Same question, same model, different answer. Nothing crashed and no warning was printed: the fact
just silently vanished. That is the danger of silent truncation.

### 4. Option B: refuse with an error (what the APIs do)
Our toy API now behaves like Anthropic's: input over the window is an error. If the input fits but
input + `max_output` does not, it starts writing and stops when the window is full.

In [6]:
class ContextWindowExceeded(Exception):
    pass

def toy_api(messages: list[dict], max_output: int, reply_tokens: int = 12) -> dict:
    n_in = count_tokens(messages)
    if n_in > WINDOW:
        raise ContextWindowExceeded(f"400 prompt is too long: {n_in} tokens > {WINDOW} maximum")
    room = WINDOW - n_in
    if reply_tokens > min(room, max_output):
        reason = "model_context_window_exceeded" if room < max_output else "max_tokens"
        return {"output_tokens": min(room, max_output), "stop_reason": reason}
    return {"output_tokens": reply_tokens, "stop_reason": "end_turn"}

print(toy_api(visible, MAX_OUTPUT))
print(toy_api(history + [question7], MAX_OUTPUT))

{'output_tokens': 12, 'stop_reason': 'end_turn'}
{'output_tokens': 12, 'stop_reason': 'end_turn'}


The trimmed chat answers normally. The full chat (177 input tokens) is also accepted: only 23 tokens are
left, but the 12-token reply squeezes in. Add Scout's reply and one more question, and the input itself overflows:

In [7]:
reply7 = {"role": "assistant", "text": toy_scout(history + [question7]), "turn": 7}
turn8 = {"role": "user", "text": "Now list every planet we talked about.", "turn": 8}
try:
    toy_api(history + [question7, reply7, turn8], MAX_OUTPUT)
except ContextWindowExceeded as err:
    print("error:", err)

error: 400 prompt is too long: 204 tokens > 200 maximum


Real providers, from their docs (see `RESEARCH.md` section 15):

| Provider (API) | Input too long | Input fits, output runs out of room |
|---|---|---|
| Anthropic (Messages) | 400 `invalid_request_error`, "prompt is too long" | Claude 4.5+: stops with `stop_reason: "model_context_window_exceeded"` |
| OpenAI (Responses) | default `truncation="disabled"`: 400 error. `truncation="auto"` drops items from the start | stops at `max_output_tokens`, `incomplete_details.reason = "max_output_tokens"` |
| Gemini | 400 `INVALID_ARGUMENT`, reported in forum threads as "input token count (N) exceeds the maximum number of tokens allowed (M)"; not on the official error page | not covered in our research; check the Gemini docs |

Gemini's wording is from community reports, not official docs, so treat it as approximate.

Silent dropping is something an app (like a chat website) or you choose to do; by default the APIs refuse.

### 5. Output and thinking share the window
The reply is written into the same window, so its tokens count too. Reasoning models also write hidden
**thinking** tokens first. On Anthropic and OpenAI these count as output (they come out of the same max output
budget and are billed as output); check your provider's docs for the exact rule. Watch thinking crowd out the visible answer:

In [8]:
def generate(thinking_needed: int, answer_needed: int, max_output: int) -> dict:
    thinking = min(thinking_needed, max_output)
    answer = min(answer_needed, max_output - thinking)
    done = answer == answer_needed
    return {"thinking": thinking, "answer": answer, "stop_reason": "end_turn" if done else "max_tokens"}

for think in [0, 20, 35]:
    print(f"thinking needs {think:2}:", generate(think, answer_needed=12, max_output=MAX_OUTPUT))

thinking needs  0: {'thinking': 0, 'answer': 12, 'stop_reason': 'end_turn'}
thinking needs 20: {'thinking': 20, 'answer': 12, 'stop_reason': 'end_turn'}
thinking needs 35: {'thinking': 35, 'answer': 5, 'stop_reason': 'max_tokens'}


With 35 tokens of thinking, only 5 are left for a 12-token answer: the user gets a cut-off reply
and `stop_reason` says `max_tokens`. Raising the max output limit helps, but every extra token you
reserve is a token the history can no longer use.

### 6. Context rot: worse answers before the limit
Fitting is not enough. Anthropic's docs: "As token count grows, accuracy and recall degrade, a phenomenon
known as context rot." One reason they give: every token can attend to every other token, so n tokens
give n² pairs to weigh. Here is how fast that grows, and how thin an even share of attention gets:

In [9]:
for n in [200, 8_000, 200_000, 1_000_000]:
    print(f"{n:>9,} tokens -> {n * n:>22,} token pairs; one token's even share = {1 / n:.7f}")

      200 tokens ->                 40,000 token pairs; one token's even share = 0.0050000
    8,000 tokens ->             64,000,000 token pairs; one token's even share = 0.0001250
  200,000 tokens ->         40,000,000,000 token pairs; one token's even share = 0.0000050
1,000,000 tokens ->      1,000,000,000,000 token pairs; one token's even share = 0.0000010


This is a picture, not a measurement: real attention is not spread evenly. But it shows the shape.
The one fact Scout needs ("My project is...") has to compete with more and more tokens around it.
Anthropic warns that accuracy and recall degrade as tokens grow, even in large windows, so a smaller,
cleaner context usually beats a bigger one.

## At work
- **Count before you send.** Teams measure every request with the provider's token counter or the
  `usage` field returned by the API, because system prompt, tool definitions and tool results all spend
  the window, not just chat text. → lesson 12
- **Overflow is an error you must handle.** A 400 "prompt is too long" or a `model_context_window_exceeded`
  stop reason is a normal production event in long chats and agent runs, not a rare crash. → lesson 13
- **Long chats need a policy, not luck.** Products trim or summarise old turns on purpose (Anthropic offers
  server-side compaction in beta, OpenAI offers `truncation="auto"`), and test what gets lost. → lesson 32
- **Context rot drives "context engineering".** Anthropic's guidance is to send the smallest set of
  high-signal tokens, not to fill a 1M-token window because you can. → lesson 31

## What just happened
- The context window counted everything: the system prompt (15 tokens), the whole history, the new question and the reserved output.
- After 6 turns Scout's chat was 168 tokens; with turn 7 and 40 reserved for output it needed 217, over the 200-token window.
- A sliding window silently dropped the user's turn-1 message, so toy Scout forgot the project.
- The toy API refused input over the window with a "prompt is too long" error, as the real APIs do by default.
- Thinking tokens came out of the same output budget and cut the visible answer short.
- Context rot: the more tokens in the window, the thinner each fact's share of attention.

## Common mistakes
**1. Checking that the input fits, but forgetting the output.** A team checks `input <= WINDOW` and
then asks for a long reply. The reply stops early because the window is full, and their code treats it as a full answer:

In [10]:
long_chat = history + [question7]
assert count_tokens(long_chat) <= WINDOW          # input fits, so "all good"
result = toy_api(long_chat, max_output=MAX_OUTPUT, reply_tokens=30)
assert result["stop_reason"] == "end_turn", f"reply cut short: {result}"

AssertionError: reply cut short: {'output_tokens': 23, 'stop_reason': 'model_context_window_exceeded'}

The reply stopped because the window was full: `model_context_window_exceeded` here, `max_tokens` when the
max output limit is hit first. Both mean the answer is incomplete.

Fix: check `input + max_output <= WINDOW` before sending (or trim until it holds), and always read the
stop reason: anything other than a normal end (`end_turn`) means the answer is incomplete.

**2. Trimming by number of messages instead of tokens.** "Keep the last 4 messages" sounds safe, until the
user pastes something long:

In [11]:
pasted = {"role": "user", "text": " ".join(["Mars has two small moons."] * 30), "turn": 8}
last_four = [SYSTEM] + (history + [pasted])[-4:]
print("messages:", len(last_four), "tokens:", count_tokens(last_four))
toy_api(last_four, MAX_OUTPUT)

messages: 5 tokens: 230


ContextWindowExceeded: 400 prompt is too long: 230 tokens > 200 maximum

Fix: budget in tokens. `sliding_window(...)` counts the cost of each message, so it keeps as many as fit,
and a single message that is too big on its own has to be shortened or rejected before you send it.

## Try it
1. Set the window to 150 tokens (keep `MAX_OUTPUT = 40`). Using `sliding_window`, which turns can the model
   still see when turn 7 is asked? What does `toy_scout` answer?
2. Write `pin_first_turn(messages, window, max_output)`: like `sliding_window`, but it always keeps the
   system prompt **and** turn 1 (both messages), then fills the rest with the newest turns.
   Check that `toy_scout` now remembers the project with `WINDOW = 200`.

In [12]:
# your code here

## Key terms
- **context window** — the most tokens a model can handle in one request: system prompt, history, new input and the output it writes (including thinking).
- **max output tokens** — the limit you set on how many tokens the model may write in one reply (`max_tokens`, `max_output_tokens`); thinking tokens count toward it.
- **thinking tokens** — hidden reasoning tokens a reasoning model writes before its answer; they use the output budget and are billed as output.
- **sliding window (truncation)** — dropping the oldest messages so the rest fits; done by apps or by you, not by most APIs by default.
- **context window exceeded** — what happens when a request does not fit: usually a 400 error ("prompt is too long"), or a reply stopped early.
- **context rot** — model accuracy and recall get worse as the number of tokens in the context grows, even below the limit.